In [1]:
import pathlib
from pathlib import Path
import pandas as pd
file_path = pathlib.Path("raw_data","wwtp_data.csv")
ad_facility = pd.read_csv(file_path)


In [2]:
df = ad_facility[["state","city","has_ad","flow_m3_per_day"]]

In [3]:
df["plant_flow_ad"] = df["flow_m3_per_day"].where(df["has_ad"]==str("yes")) 

In [4]:
df = df.dropna(axis=0,how="any")

In [5]:
#physical constants
MMBTU_TO_MJ = 1055.06 # 1mmBtu = 1055.06MJ a conversion 
HHV1 = 55.6 # MJ/KgCH4 (El Abbadi, Preprint)
eff = 0.90 # Methane recovery effficiency (%) from Parker etal.
mc = 0.65 # Methane content in biogas (%) El Abbadi preprint
p = 1.15 # Density of biogas in kg/m3(Jørgensen,2009)​
p1 = 0.656 # Density of methane in kg/m3(Speight,2020)​


In [6]:
#tecno-economic assumptions
cf = 8322 # 95% Capacity Factor(cf) in hours
t = 20 # lifetime technology in years 
wacc = 0.1 # WACC is the Weighted Average Cost of Capital (WACC). A 10% WACC is assumed. Will refine later

In [7]:
def calc_biogas_flow(df): 
    """Calculates biogas flowrate in KgCH4/h from El Abbadi preprint"""
    df["biogas_flow_kg_h"] = df["plant_flow_ad"]*0.00148
    return df

df = calc_biogas_flow(df)


In [8]:
def calc_biogas_flow_m3(df,p):
    """Calculates biogas flowrate in m3/h"""
    df["capacity_m3_h"] = df["biogas_flow_kg_h"]/p
    return df

df = calc_biogas_flow_m3(df,p)

In [9]:
def calc_rng_flow_m3(df,mc,eff):
    """Calculates RNG flow in m3/h"""
    df["rng_m3_h"] = df["capacity_m3_h"]*mc*eff
    return df

df = calc_rng_flow_m3(df,mc,eff)

In [10]:
def calc_rng_flow_mmbtu(df,p1,HHV1,MMBTU_TO_MJ):
    """Calculates RNG flow in mmbtu/h"""
    df["rng_mmbtu_h"] = (df["rng_m3_h"]*p1*HHV1)/MMBTU_TO_MJ
    return df

df = calc_rng_flow_mmbtu(df,p1, HHV1 , MMBTU_TO_MJ)

In [11]:
def calc_capex(df):
    """Calculates Capex($) using Parker etal model"""
    df["Capex($)"] = ((df["rng_mmbtu_h"])**0.56)*1370000
    return df

df = calc_capex(df)